In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

elevation = dataset.select('DEM')

elev_projection = elevation.first().projection()

roads = ee.FeatureCollection("projects/deforestation-495419/assets/Panama_OSM_Roads")

roads_raster = ee.Image().byte().paint(roads, 1)

distance_roads = (
    roads_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .rename("dist_roads_m")
    .clip(panama_geom)
)

# reprojecting to fit other data layers
# sample_image = elevation
# collection_projection = sample_image.projection()

# reprojected_distance_roads = (
#     distance_roads.resample("bilinear")
#     .reproject(collection_projection)
#     .rename("dist_roads_reprojected")
#     .clip(panama_geom)
# )

# Map.addLayer(roads_raster, {"palette": ["black"]}, "Roads")
# Map.addLayer(distance_roads, {"min": 0, "max": 5000}, "Dist Roads")
# Map.addLayer(reprojected_distance_roads, {"min": 0, "max": 5000}, "Dist Roads Reprojected")

# Map

In [6]:
# export this as an image asset with the elev_projection and scale.
task = ee.batch.Export.image.toAsset(
    image = distance_roads,
    description = 'distance_roads',
    assetId = "projects/deforestation-panama/assets/distance_roads",
    region = panama_geom,
    scale = elev_projection.nominalScale(),
    crs = elev_projection.crs(),
    crsTransform = elev_projection.getInfo()['transform'],
    maxPixels=1e12
)

task.start()

In [2]:
Map = geemap.Map()
Map.centerObject(panama_geom, 7)

export = ee.Image("projects/deforestation-panama/assets/distance_roads")

Map.addLayer(roads_raster, {"palette": ["black"]}, "Roads")
Map.addLayer(distance_roads, {"min": 0, "max": 5000}, "Dist Roads")
Map.addLayer(export, {}, "Dist roads reproj")

Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…

RefreshError: ('invalid_grant: Token has been expired or revoked.', {'error': 'invalid_grant', 'error_description': 'Token has been expired or revoked.'})

RefreshError: ('invalid_grant: Token has been expired or revoked.', {'error': 'invalid_grant', 'error_description': 'Token has been expired or revoked.'})